# 04 — The checkout funnel

Phase 3 deliverable A. Every metric in Phase 1 §5.3.

**The opening argument of the project is one number pair:** checkout conversion
versus net conversion. Checkout conversion is what a checkout team is usually
measured on. Net conversion is what the business banks. Everything else in this
notebook explains the distance between them.

Mirrors `sql/analysis/10_funnel.sql` (Q1–Q4). `scripts/05_crosscheck.py` asserts the two
agree — this notebook does not re-derive that, it relies on it.


In [ ]:
import sys, json; from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.analysis import funnel as F

TABLES = F.load_tables()
TRUTH  = json.loads((ROOT / 'data/truth/_truth.json').read_text())
pd.set_option('display.width', 160)
print({k: len(v) for k, v in TABLES.items()})


## Q1 — the funnel, end to end


In [ ]:
steps = F.funnel_steps(TABLES)
steps['step_conversion'] = steps['n'] / steps['n'].shift(1)
steps


## Q2 — the two conversions, and the CM/CSS baseline


In [ ]:
conv = F.conversions(TABLES)
pd.Series(conv)


In [ ]:
leak_pp = conv['conversion_leak'] * 100
print(f"checkout conversion   {conv['checkout_conversion']:.2%}")
print(f"net conversion        {conv['net_conversion']:.2%}")
print(f"LEAK                  {leak_pp:.2f}pp of every session started")
print()
print(f"censored (unresolved) {conv['censored_excluded']:,} orders --"
      f" neither delivered nor RTO")
print(f"net conversion, resolved basis  "
      f"{conv['net_conversion_resolved_basis']:.2%}")


The resolved-basis figure is the honest one to quote for a steady state: censoring
is an artefact of a 90-day observation window, not a real-world outcome. Both are
reported; neither replaces the other (limitation L9).


## Q3 — where sessions die


In [ ]:
F.abandon_steps(TABLES)


`FEE_REVEAL` is absent, and that is expected: no shipping fee is charged in the
baseline, so the branch has nothing to fire on. It is the diagnosis waiting for a
fee intervention, not a dead code path (decision A25).


## Q4 — payment reliability, and the COD it manufactures


In [ ]:
pay = F.payment_success(TABLES)
pd.Series(pay)


In [ ]:
print(f"prepaid success rate      {pay['prepaid_success_rate']:.2%}")
print(f"COD orders from a failure {pay['pct_of_cod_from_failure']:.2%}"
      f"  <- H11, prior was 8-15%")
print(f"sessions lost at failure  {pay['abandoned_at_failure']:,}")


> COD has no payment-success analogue. These are prepaid-only figures and must
> never be blended with COD (Phase 1 §5.3).
